# Ultimate judgement

**COSC2753 Assignment 2 - Fashion Intelligence System**

Four models were built over the same catalogue of 60x80 product photographs: item
type, season, gender and usage, and visual search. Each was selected and reported
inside its own notebook, against its own held-out split. This notebook asks the
question none of them can answer alone.

## The judgement

> **The system is fit to deploy against catalogue-style product photography, and
> is not fit to deploy against unconstrained user photographs without an explicit
> confidence gate. The binding constraint is not model capacity - it is the gap
> between the flat-lay images the models were trained on and the images a user
> actually submits.**

Two claims, and both are testable. The first says the models clear a useful bar
on the distribution they were built for. The second says they degrade on a
different distribution in a way that is predictable, measurable, and detectable
at inference time - which is what makes a gate possible rather than a hope.

Everything below is computed from the artefacts each task wrote. Nothing is
restated from memory.

In [ ]:
# ============================================
# CELL 1 - Setup
# ============================================

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd() if (Path.cwd() / "artifacts").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))

ARTIFACTS = PROJECT_DIR / "artifacts"
OUTPUTS = PROJECT_DIR / "outputs"

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)


def load_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else {}


def load_csv(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


task1 = load_json(ARTIFACTS / "task1" / "task1_summary.json")
task2 = load_json(ARTIFACTS / "task2" / "task2_season_metrics.json")
task3 = load_json(ARTIFACTS / "task3" / "task3_cnn_summary.json")
task4 = load_json(ARTIFACTS / "task4" / "task4_summary.json")
manifest = load_json(ARTIFACTS / "task4" / "search_manifest.json")

print("Loaded summaries for", sum(bool(d) for d in (task1, task2, task3, task4)),
      "of 4 tasks")

## 1. What each model achieves on the distribution it was built for

The four tasks are not comparable on a single number - a 92-class problem, a
4-class problem, two 5-and-4-class problems and a ranking problem have different
ceilings and different floors. What makes them comparable is the *margin over the
naive answer*: how much better is the model than always predicting the majority
class, or than returning random items?

That margin is the honest common currency, and it is what the table below
reports.

In [ ]:
# ============================================
# CELL 2 - Headline performance, and the margin over doing nothing
# ============================================

rows = [
    {
        "task": "1 - item type",
        "target": "articleType (92 classes)",
        "metric": "weighted F1",
        "model": task1.get("test_weighted_f1_deployed"),
        "naive": task1.get("baseline_weighted_f1"),
    },
    {
        "task": "2 - season",
        "target": "season (4 classes)",
        "metric": "macro F1",
        "model": task2.get("test", {}).get("macro_f1", np.nan) * 100,
        "naive": task2.get("baseline", {}).get("macro_f1", np.nan) * 100,
    },
    {
        "task": "3 - gender",
        "target": "gender (5 classes)",
        "metric": "accuracy",
        "model": task3.get("test_gender_accuracy"),
        "naive": task3.get("baseline_gender_accuracy"),
    },
    {
        "task": "3 - usage",
        "target": "usage (4 classes)",
        "metric": "accuracy",
        "model": task3.get("test_usage_accuracy"),
        "naive": task3.get("baseline_usage_accuracy"),
    },
    {
        "task": "4 - visual search",
        "target": "top-10 retrieval",
        "metric": "P@10",
        "model": task4.get("clean_P@10"),
        "naive": task4.get("random_baseline_P@10"),
    },
]

headline = pd.DataFrame(rows)
headline["margin"] = (headline["model"] - headline["naive"]).round(2)
headline["model"] = headline["model"].round(2)
headline["naive"] = headline["naive"].round(2)
display(headline)

print("Every task clears its naive floor by a wide margin. Task 2 clears it by the")
print("least, and its ceiling is the lowest - season is only weakly determined by")
print("a 60x80 photograph, because a black t-shirt is worn all year.")

## 2. The failure they share

Each notebook reports a domain gap of its own. Read together they are plainly the
same gap, and that is the finding this section exists to establish: the four
models fail in the same way, at the same boundary, for the same reason.

Task 1 measured it by corrupting its inputs. Task 4 measured it by compositing
held-out items onto backgrounds the encoder had never seen. Neither was designed
to test the other, and they agree.

In [ ]:
# ============================================
# CELL 3 - The same collapse, measured twice, independently
# ============================================

ood = load_csv(OUTPUTS / "evaluation" / "task1_ood_results.csv")
disjoint = load_csv(OUTPUTS / "task4_disjoint_benchmark.csv")

gap_rows = []

if not ood.empty:
    deployed = ood[ood["checkpoint"].str.contains("task1_cnn", na=False)]
    for ingest in deployed["ingest"].unique():
        subset = deployed[deployed["ingest"] == ingest]
        clean = subset[subset["severity"] == "clean"]["accuracy"]
        worst = subset[subset["severity"] != "clean"]["accuracy"]
        if len(clean) and len(worst):
            gap_rows.append({
                "task": "1 - item type",
                "measured by": f"corrupted inputs ({ingest})",
                "in-domain": round(float(clean.iloc[0]), 2),
                "out-of-domain": round(float(worst.min()), 2),
                "retained %": round(float(worst.min()) / float(clean.iloc[0]) * 100, 1),
            })

if not disjoint.empty:
    for model in disjoint["model"].unique():
        subset = disjoint[disjoint["model"] == model]
        clean = subset[subset["benchmark"] == "clean"]["P@10"]
        hard = subset[subset["benchmark"].str.contains("disjoint")]["P@10"]
        if len(clean) and len(hard):
            gap_rows.append({
                "task": f"4 - visual search ({model})",
                "measured by": "composited backgrounds",
                "in-domain": round(float(clean.iloc[0]), 2),
                "out-of-domain": round(float(hard.iloc[0]), 2),
                "retained %": round(float(hard.iloc[0]) / float(clean.iloc[0]) * 100, 1),
            })

gaps = pd.DataFrame(gap_rows)
display(gaps)

print("The pattern is consistent and it is severe. A model trained only on flat-lay")
print("catalogue photographs does not degrade gracefully when the input stops being")
print("one - it collapses. Task 4's clean encoder retains about a seventh of its")
print("precision; Task 1 under corruption retains about a tenth of its accuracy.")

### The gap is a property of the training distribution, not of the architecture

The strongest evidence that this is a data problem rather than a capacity problem
comes from the one intervention that addressed the distribution directly.

Task 4 retrained its encoder with items composited onto random backgrounds at
random scales - no change to the architecture, no extra parameters, the same
128-dimensional output. The result is not a marginal gain.

In [ ]:
# ============================================
# CELL 4 - Changing the training distribution, not the model
# ============================================

if not disjoint.empty:
    pivot = disjoint.pivot_table(index="model", columns="benchmark", values="P@10")
    display(pivot.round(2))

    clean_col = "clean"
    hard_col = [c for c in pivot.columns if "disjoint" in c][0]
    baseline = pivot.loc[[i for i in pivot.index if "clean baseline" in i][0]]
    augmented = pivot.loc[[i for i in pivot.index if "bg-aug" in i][0]]

    print(f"in-domain cost of the change : {augmented[clean_col] - baseline[clean_col]:+.2f} P@10")
    print(f"out-of-domain gain           : {augmented[hard_col] - baseline[hard_col]:+.2f} P@10")
    print()
    print("Roughly one point of in-domain precision buys nearly fifty points out of")
    print("domain, from an architecturally identical model. That asymmetry is the")
    print("whole argument: the models are not short of capacity, they are short of")
    print("the right training distribution.")

## 3. Independent evaluation

Two forms of independence are worth separating.

**Data the models never saw, from outside the dataset entirely.** The 31
photographs in `A2_FashionDataset/input_images` were collected from the open web:
mixed formats, cluttered backgrounds, worn garments, several items per frame, and
some non-clothing. They share no provenance with the supplied catalogue and no
label with it either, which is exactly what makes them useful and what limits
what they can prove.

**A reference implementation trained by someone else.** Task 1 was additionally
compared against a pretrained backbone fine-tuned on the same split, which sets a
bar the from-scratch network had to clear.

In [ ]:
# ============================================
# CELL 5 - Against a pretrained reference, and against real photographs
# ============================================

reference = load_json(ARTIFACTS / "task1" / "pretrained_reference.json")
if reference:
    comparison = pd.DataFrame([
        {"model": "Task 1 CNN, trained from scratch",
         "test accuracy": task1.get("test_accuracy_deployed"),
         "test weighted F1": task1.get("test_weighted_f1_deployed"),
         "test macro F1": task1.get("test_macro_f1_deployed")},
        {"model": "pretrained backbone, same split",
         "test accuracy": reference["test"]["accuracy"],
         "test weighted F1": reference["test"]["weighted_f1"],
         "test macro F1": reference["test"]["macro_f1"]},
    ]).round(2)
    display(comparison)
    delta = (task1.get("test_weighted_f1_deployed", 0)
             - reference["test"]["weighted_f1"])
    print(f"The from-scratch network is {delta:+.2f} weighted F1 against the pretrained")
    print("reference. At 60x80 the features that transfer from natural-image")
    print("pretraining are largely the ones the resolution has already destroyed, so")
    print("this is less surprising than it first looks - and it is the reason the")
    print("submitted models are all trained from scratch.")

print()
domain = load_csv(OUTPUTS / "task4_domain_comparison.csv")
clusters = load_csv(OUTPUTS / "cluster_domain_comparison.csv")
if not domain.empty:
    print("Retrieval behaviour on catalogue images vs collected web photographs:")
    display(domain)
if not clusters.empty:
    print("Cluster assignment confidence on the same two populations:")
    display(clusters)

### What the collected photographs can and cannot show

They carry no ground-truth labels, so they cannot produce an accuracy figure. What
they can produce is a *distributional* comparison, and that is enough for the
judgement being made: on catalogue-style inputs the retrieved neighbourhood is
tight and internally consistent, and on web photographs it is neither.

That is the second claim of the judgement, and it is now measured rather than
asserted. It also points directly at the mechanism for acting on it - if the
degradation is visible in the retrieved neighbourhood, it is visible at inference
time, without labels.

The two tables above were produced by different notebooks at different times, so
their sample sizes differ: the retrieval comparison covers all 31 collected
photographs, while the cluster comparison covers the 19 that existed when that
notebook last ran. Re-running `06_task4_clustering.ipynb` brings the second onto
the full set. The direction of the difference is the same in both, and it is the
direction that carries the argument.

## 4. The degradation is detectable at inference time

A system that fails on out-of-distribution input but cannot tell that it is
failing is unusable in production. A system that fails and *knows* it is failing
can decline, ask for a better photograph, or route to a human.

Two label-free signals were already available and are now surfaced by the API:
the top-1 similarity, and the coherence of the returned set - the share of the
top-K agreeing with the top result's type. Both separate the two populations.

In [ ]:
# ============================================
# CELL 6 - Confidence signals over the whole test set
# ============================================

test_summary = load_json(OUTPUTS / "task4_test_summary.json")
fallback = load_csv(OUTPUTS / "task4_ingestion_fallback.csv")

if test_summary:
    print(f"Test images scored          : {test_summary['images']:,}")
    print(f"Mean top-1 similarity       : {test_summary['mean_top1_similarity']:.3f}")
    print(f"Mean coherence              : {test_summary['mean_coherence']:.3f}")
    print(f"Answered confidently        : {test_summary['confident_share']:.1%}")
    print(f"Cluster-confident            : {test_summary.get('cluster_confident_share', float('nan')):.1%}")
    print()
    print("About seven in ten catalogue-style images are answered confidently under")
    print("thresholds calibrated to sit between the two populations. The remainder")
    print("are not wrong by definition - they are the queries the system should not")
    print("assert an answer for without review.")

if not fallback.empty:
    print()
    print("Ingestion behaviour, and whether declining to segment costs anything:")
    display(fallback[["ingestion", "images", "mean_top1_similarity",
                      "mean_coherence", "confident_share"]])

## 5. Where the system fails, stated plainly

Four limitations are load-bearing enough that a deployment decision has to
account for them.

**Resolution is the ceiling.** At 60x80 a lipstick and a deodorant occupy the same
few hundred pixels. Task 3's gender head systematically predicts `Men` for makeup
items, and no amount of training fixes a distinction the input does not contain.

**Season is weakly determined.** Task 2 clears its naive floor by the smallest
margin of the four, because the target is only loosely a property of the image.

**The long tail stays weak.** Task 1's macro F1 sits far below its weighted F1:
the headline is carried by common classes while rare ones remain unreliable. The
32 classes dropped for having too few examples are a deliberate scope reduction,
not a solved problem.

**One vector per image.** Task 4 embeds a whole frame, so a photograph of a person
wearing several garments is answered as though it were one item. Region search
mitigates this and does not remove it.

In [ ]:
# ============================================
# CELL 7 - The long tail, quantified
# ============================================

print("Task 1")
print(f"  weighted F1 {task1.get('test_weighted_f1_deployed', float('nan')):.2f}"
      f"   macro F1 {task1.get('test_macro_f1_deployed', float('nan')):.2f}"
      f"   gap {task1.get('test_weighted_f1_deployed', 0) - task1.get('test_macro_f1_deployed', 0):.2f}")
print(f"  classes kept {task1.get('classes_after_drop')} of {task1.get('classes_before_drop')}"
      f"  ({task1.get('classes_dropped')} dropped as too rare)")
print(f"  train accuracy {task1.get('train_accuracy', float('nan')):.2f}"
      f"  vs test {task1.get('test_accuracy_deployed', float('nan')):.2f}"
      f"  (gap {task1.get('train_test_gap', float('nan')):.2f})")

print()
print("Task 3")
print(f"  gender  accuracy {task3.get('test_gender_accuracy', float('nan')):.2f}"
      f"   macro F1 {task3.get('test_gender_macro_f1', float('nan')):.2f}")
print(f"  usage   accuracy {task3.get('test_usage_accuracy', float('nan')):.2f}"
      f"   macro F1 {task3.get('test_usage_macro_f1', float('nan')):.2f}")
print(f"  both correct on the same item: {task3.get('test_exact_match', float('nan')):.2f}%")
print(f"  measured noise floor: {task3.get('noise_floor', float('nan')):.3f} macro F1")

print()
print("Every one of these gaps is a long-tail gap. The systems are reliable on the")
print("classes the catalogue is actually made of, and unreliable on the ones it")
print("barely contains.")

## 6. The operating policy this implies

The judgement is conditional, so the conditions have to be written down.

| Input | Decision | Basis |
|---|---|---|
| Catalogue-style product photograph | Serve all four predictions | in-domain figures in section 1 |
| User photograph, confident signals | Serve, with the confidence shown | section 4 separates the populations |
| User photograph, low confidence | Decline, or ask for a plain-background photo | section 2 shows the collapse is severe, not gradual |
| Several garments in frame | Return per-region results, not one answer | section 5 |

This is why the deployed encoder is the background-augmented one despite it being
*worse* in domain. Selecting on the clean benchmark would have picked a model that
scores about a point higher on images the product will rarely see, and collapses
on the images it will actually receive.

The same reasoning is why the confidence gate is part of the system rather than a
nicety: the honest answer to a hard query is that the system does not know.

## 7. The judgement, restated

The system meets a useful bar on the distribution it was built for. Item type,
gender and usage are all well above their naive floors and are usable as catalogue
tooling. Season is the weakest and should be presented as a suggestion rather than
a label. Visual search returns a same-type item in roughly four of five top-10
slots on catalogue queries.

It does not meet that bar on unconstrained photographs, and the shortfall is large
enough that presenting the same interface for both inputs would be misleading. The
mitigation is not a better architecture - background augmentation already bought
more out-of-domain precision than any architectural change in this project, at a
cost of about one in-domain point - but a constrained input and an explicit gate.

**Deploy against catalogue photography. Gate everything else.**

The single change that would move this conclusion most is not a larger model. It
is higher-resolution source images: every limitation in section 5 traces back to
what 60x80 pixels can represent.